# SAM2 選手トラッキング（Colab / GPU）

交差に強いピクセル追跡。流れ:
1. 動画の一区間を高画質フレームに切り出し
2. YOLOで先頭フレームの選手ボックスを自動取得 → SAM2の初期プロンプトに
3. SAM2で全フレームへ伝播（オクルージョン/交差に強い）
4. ローカルGUIが読める `tracks.csv` + フレームzip を Drive に保存

**ランタイム → ランタイムのタイプを変更 → GPU(T4/A100)** にしてから実行。

In [ ]:
# 1) SAM2 と YOLO のインストール
!git clone -q https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -q -e . >/dev/null 2>&1
!pip install -q ultralytics >/dev/null 2>&1
# モデルサイズ: tiny / small / base_plus / large。22人を回すので small 推奨（large の3〜4倍速）
SAM2_SIZE = 'small'   # 速度優先。精度を上げたいなら 'large'
_CKPT = {'tiny':'sam2.1_hiera_tiny.pt','small':'sam2.1_hiera_small.pt',
         'base_plus':'sam2.1_hiera_base_plus.pt','large':'sam2.1_hiera_large.pt'}[SAM2_SIZE]
_CFG  = {'tiny':'sam2.1_hiera_t.yaml','small':'sam2.1_hiera_s.yaml',
         'base_plus':'sam2.1_hiera_b+.yaml','large':'sam2.1_hiera_l.yaml'}[SAM2_SIZE]
!mkdir -p checkpoints && wget -q -O checkpoints/{_CKPT} https://dl.fbaipublicfiles.com/segment_anything_2/092824/{_CKPT}
print('インストール完了  モデル=', SAM2_SIZE)

In [ ]:
# 2) Drive をマウントして動画パスとパラメータを設定
from google.colab import drive
drive.mount('/content/drive')

# ↓↓ ここを自分の環境に合わせる ↓↓
VIDEO   = '/content/drive/MyDrive/緑黒1試合目.MOV'  # Driveに置いた動画
NAME    = 'seg_sam_105'      # セグメント名（ローカルの outputs/segments/<NAME> になる）
START_SEC = 105.0            # 開始秒（1:45）
SECONDS   = 30.0             # 区間長。SAM2はメモリを食うのでまず30秒程度から
FPS       = 8.0             # SAM2は低fpsでも交差に強い。8前後で十分
OUT_W, OUT_H = 1920, 1080
OUT_DIR = f'/content/out_{NAME}'
FRAMES_DIR = f'{OUT_DIR}/frames'
import os; os.makedirs(FRAMES_DIR, exist_ok=True)
print('設定OK', VIDEO)

In [ ]:
# 3) フレーム切り出し（ffmpegでHEVC(.MOV)もデコード。00000.jpg連番・1920幅）
import os, glob, subprocess
assert os.path.exists(VIDEO), f'動画が見つからない: {VIDEO}  ← Driveのパスを確認'
for p in glob.glob(f'{FRAMES_DIR}/*.jpg'):
    os.remove(p)
cmd = ['ffmpeg','-y','-loglevel','error','-ss',str(START_SEC),'-i',VIDEO,
       '-t',str(SECONDS),'-vf',f'fps={FPS},scale={OUT_W}:-2',
       '-q:v','3','-start_number','0',f'{FRAMES_DIR}/%05d.jpg']
subprocess.run(cmd, check=True)
N_FRAMES = len(glob.glob(f'{FRAMES_DIR}/*.jpg'))
EFF_FPS = FPS
assert N_FRAMES > 0, 'フレームが0枚。動画パス/コーデックを確認'
print(f'実効fps={EFF_FPS}  フレーム数={N_FRAMES}')

In [ ]:
# 4) YOLOモデルをロード（初期シード＆定期再シード用）
from ultralytics import YOLO
import numpy as np
ydet = YOLO('yolo11m.pt')

def yolo_boxes(frame_idx):
    r = ydet.predict(f'{FRAMES_DIR}/{frame_idx:05d}.jpg', classes=[0],
                     imgsz=1280, conf=0.25, verbose=False)[0]
    return r.boxes.xyxy.cpu().numpy()

print('先頭フレーム検出数:', len(yolo_boxes(0)))

In [ ]:
# 5) SAM2 + YOLO定期再シード
#    キーフレームごとに、SAM2の各オブジェクトをYOLO検出へ「一意マッチ」して再アンカー。
#    → 2オブジェクトが同じ選手に吸い付く重複を構造的に禁止／ドリフトを立て直す／途中入場を新規追加。
import torch
from scipy.optimize import linear_sum_assignment
from sam2.build_sam import build_sam2_video_predictor

KEYFRAME_EVERY = 8     # 何フレームごとにYOLOで答え合わせするか（8fpsなら約1秒）
MATCH_DIST     = 80    # 足元距離(px,1920系)。これ以内なら同一とみなす

cfg = f'configs/sam2.1/{_CFG}'
ckpt = f'checkpoints/{_CKPT}'
predictor = build_sam2_video_predictor(cfg, ckpt, device='cuda')

def foot(b):
    return ((b[0] + b[2]) / 2.0, b[3])

def masks_to_boxes(obj_ids, mask_logits):
    """全オブジェクトのlogitをピクセルargmaxで排他化→各オブジェクトのbbox。"""
    ml = mask_logits.squeeze(1)              # [num_obj, H, W]
    maxv, argm = ml.max(0)
    out = {}
    for k, oid in enumerate(obj_ids):
        m = ((argm == k) & (maxv > 0)).cpu().numpy()
        ys, xs = np.where(m)
        if len(xs) < 10:
            continue
        out[int(oid)] = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
    return out

state = predictor.init_state(video_path=FRAMES_DIR)
next_oid = 1
for b in yolo_boxes(0):                       # 先頭をシード
    predictor.add_new_points_or_box(state, frame_idx=0, obj_id=next_oid,
                                    box=b.astype(np.float32)); next_oid += 1

seg = {}
cur = 0
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
    while cur < N_FRAMES - 1:
        kf = min(cur + KEYFRAME_EVERY, N_FRAMES - 1)
        for f_idx, obj_ids, ml in predictor.propagate_in_video(
                state, start_frame_idx=cur, max_frame_num_to_track=kf - cur):
            seg[f_idx] = masks_to_boxes(obj_ids, ml)

        # --- キーフレームkfでYOLO再シード ---
        dets = yolo_boxes(kf)
        cur_boxes = seg.get(kf, {})
        oids = list(cur_boxes.keys())
        used = set()
        if oids and len(dets):
            of = [foot(cur_boxes[o]) for o in oids]
            df = [foot(d) for d in dets]
            C = np.array([[((a[0]-b[0])**2 + (a[1]-b[1])**2)**0.5 for b in df] for a in of])
            ri, cj = linear_sum_assignment(C)
            for i, j in zip(ri, cj):
                if C[i, j] <= MATCH_DIST:
                    predictor.add_new_points_or_box(state, frame_idx=kf, obj_id=oids[i],
                                                    box=dets[j].astype(np.float32))
                    used.add(j)
        for j, d in enumerate(dets):           # 未マッチ検出＝途中入場 → 新規オブジェクト
            if j not in used:
                predictor.add_new_points_or_box(state, frame_idx=kf, obj_id=next_oid,
                                                box=d.astype(np.float32)); next_oid += 1
        cur = kf
print(f'伝播完了: {len(seg)}フレーム  総オブジェクト数={next_oid-1}')

In [ ]:
# 6) tracks.csv に書き出し（ローカルGUIと同形式。pitch座標はローカルで付与）
import csv
with open(f'{OUT_DIR}/tracks.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['frame','track_id','x1','y1','x2','y2','foot_x','foot_y','x_pitch','y_pitch','in_pitch','conf'])
    for f_idx in sorted(seg):
        for oid, (x1,y1,x2,y2) in seg[f_idx].items():
            fx, fy = (x1+x2)/2, y2
            w.writerow([f_idx, oid, x1, y1, x2, y2, round(fx,1), round(fy,1), '', '', 0, 1.0])

import json
json.dump({'name':NAME,'start_sec':START_SEC,'seconds':SECONDS,
           'eff_fps':EFF_FPS,'n_frames':N_FRAMES,'width':OUT_W,'height':OUT_H},
          open(f'{OUT_DIR}/meta.json','w'), ensure_ascii=False, indent=2)
print('tracks.csv / meta.json 作成')

In [ ]:
# 7) フレーム+csv を zip して Drive に保存（ローカルにDLして ingest_sam2.py にかける）
import shutil
zip_path = f'/content/drive/MyDrive/{NAME}.zip'
shutil.make_archive(f'/content/{NAME}', 'zip', OUT_DIR)
shutil.copy(f'/content/{NAME}.zip', zip_path)
print('保存:', zip_path)
print('→ これをローカルにDLし、解凍して  .venv/bin/python ingest_sam2.py --name', NAME, ' --dir <解凍先>')